In [ ]:
```xml
<VSCode.Cell language="markdown">
# 💰 Cloud Cost Optimization & FinOps Guide

**Financial operations for cloud infrastructure, cost allocation, and optimization**

This guide covers:
- Cost analysis and attribution
- Resource optimization strategies
- Reserved instances and committed use discounts
- Spot instances and preemptible VMs
- Cost forecasting and budgeting
- Chargeback and showback models
- Cost anomaly detection
- Optimization recommendations
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Cost Analysis & Tracking

### Cost Center Attribution
```python
from datetime import datetime, timedelta
from enum import Enum

class ResourceTag(Enum):
    ENVIRONMENT = "environment"
    TEAM = "team"
    PROJECT = "project"
    COST_CENTER = "cost_center"
    APPLICATION = "application"

class CostAllocation:
    """Track costs by allocation keys"""
    
    def __init__(self):
        self.resources = {}
        self.costs = {}
    
    def register_resource(self, resource_id: str, tags: dict):
        """Register resource with tags"""
        self.resources[resource_id] = {
            'tags': tags,
            'created_at': datetime.now()
        }
    
    def record_cost(self, resource_id: str, cost: float, timestamp: datetime = None):
        """Record resource cost"""
        timestamp = timestamp or datetime.now()
        
        if resource_id not in self.costs:
            self.costs[resource_id] = []
        
        self.costs[resource_id].append({
            'cost': cost,
            'timestamp': timestamp
        })
    
    def get_costs_by_team(self) -> dict:
        """Aggregate costs by team"""
        costs_by_team = {}
        
        for resource_id, resource_data in self.resources.items():
            team = resource_data['tags'].get('team', 'unassigned')
            
            if team not in costs_by_team:
                costs_by_team[team] = 0
            
            if resource_id in self.costs:
                team_total = sum(entry['cost'] for entry in self.costs[resource_id])
                costs_by_team[team] += team_total
        
        return costs_by_team
    
    def get_costs_by_project(self) -> dict:
        """Aggregate costs by project"""
        costs_by_project = {}
        
        for resource_id, resource_data in self.resources.items():
            project = resource_data['tags'].get('project', 'unassigned')
            
            if project not in costs_by_project:
                costs_by_project[project] = 0
            
            if resource_id in self.costs:
                project_total = sum(entry['cost'] for entry in self.costs[resource_id])
                costs_by_project[project] += project_total
        
        return costs_by_project
    
    def get_cost_trend(self, resource_id: str, days: int = 30) -> list:
        """Get cost trend over period"""
        if resource_id not in self.costs:
            return []
        
        cutoff = datetime.now() - timedelta(days=days)
        trend = [entry for entry in self.costs[resource_id] 
                if entry['timestamp'] >= cutoff]
        
        return sorted(trend, key=lambda x: x['timestamp'])

# Usage
allocator = CostAllocation()

allocator.register_resource('vm-001', {
    'team': 'backend',
    'project': 'api-gateway',
    'cost_center': 'platform'
})

allocator.record_cost('vm-001', 45.50)
allocator.record_cost('vm-001', 47.20)

print("Costs by team:", allocator.get_costs_by_team())
```

### Cost Forecasting
```python
import numpy as np
from sklearn.linear_model import LinearRegression

class CostForecaster:
    """Forecast future costs"""
    
    def __init__(self):
        self.historical_costs = []
    
    def add_observation(self, date: datetime, cost: float):
        """Add historical cost observation"""
        self.historical_costs.append({
            'date': date,
            'cost': cost,
            'days_since_start': (date - self.historical_costs[0]['date']).days 
                               if self.historical_costs else 0
        })
    
    def forecast_next_30_days(self) -> float:
        """Forecast average daily cost for next 30 days"""
        
        if len(self.historical_costs) < 2:
            return 0
        
        # Prepare data
        X = np.array([[obs['days_since_start']] for obs in self.historical_costs])
        y = np.array([obs['cost'] for obs in self.historical_costs])
        
        # Fit linear regression
        model = LinearRegression()
        model.fit(X, y)
        
        # Predict next 30 days
        last_day = X[-1][0]
        predictions = []
        
        for i in range(1, 31):
            future_day = last_day + i
            predicted_cost = model.predict([[future_day]])[0]
            predictions.append(predicted_cost)
        
        return np.mean(predictions)
    
    def get_monthly_forecast(self) -> float:
        """Get forecast for next month"""
        avg_daily = self.forecast_next_30_days()
        return avg_daily * 30
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Optimization Strategies

### Reserved Instances & Commitments
```python
class InstanceOptimizer:
    """Recommend reserved instances and committed use discounts"""
    
    def __init__(self):
        self.on_demand_instances = []
    
    def add_on_demand_instance(self, instance_id: str, instance_type: str, 
                              hourly_cost: float, uptime_days: int):
        """Register on-demand instance"""
        self.on_demand_instances.append({
            'id': instance_id,
            'type': instance_type,
            'hourly_cost': hourly_cost,
            'uptime_days': uptime_days
        })
    
    def get_optimization_recommendations(self) -> list:
        """Get RI and CUD recommendations"""
        recommendations = []
        
        for instance in self.on_demand_instances:
            # Instances running > 30 days are candidates for RIs
            if instance['uptime_days'] > 30:
                monthly_cost = instance['hourly_cost'] * 24 * 30
                
                # Estimate 1-year RI discount (typically 30-40%)
                ri_monthly_cost = monthly_cost * 0.65
                monthly_savings = monthly_cost - ri_monthly_cost
                annual_savings = monthly_savings * 12
                
                if annual_savings > 100:  # Only recommend if >$100/year savings
                    recommendations.append({
                        'instance_id': instance['id'],
                        'type': 'reserved_instance_1yr',
                        'current_annual_cost': monthly_cost * 12,
                        'ri_annual_cost': ri_monthly_cost * 12,
                        'estimated_annual_savings': annual_savings,
                        'uptime_percentage': (instance['uptime_days'] / 365) * 100
                    })
        
        return sorted(recommendations, 
                     key=lambda x: x['estimated_annual_savings'], 
                     reverse=True)

# Usage
optimizer = InstanceOptimizer()
optimizer.add_on_demand_instance('vm-001', 't2.large', 0.10, 100)
optimizer.add_on_demand_instance('vm-002', 't2.medium', 0.05, 15)

recommendations = optimizer.get_optimization_recommendations()
for rec in recommendations:
    print(f"Reserve {rec['instance_id']}: Save ${rec['estimated_annual_savings']:.2f}/year")
```

### Spot Instance Strategy
```python
class SpotInstanceManager:
    """Manage spot instances for cost optimization"""
    
    def __init__(self, interruption_threshold: float = 0.05):
        self.interruption_threshold = interruption_threshold
        self.workloads = []
    
    def add_workload(self, name: str, resilient: bool, max_interruption_rate: float):
        """Register workload"""
        self.workloads.append({
            'name': name,
            'resilient': resilient,
            'max_interruption_rate': max_interruption_rate
        })
    
    def recommend_spot_eligibility(self) -> list:
        """Recommend workloads for spot instances"""
        recommendations = []
        
        for workload in self.workloads:
            if workload['resilient'] and \
               workload['max_interruption_rate'] >= self.interruption_threshold:
                
                # Estimate 70% savings with spot
                estimated_savings_pct = 0.70
                
                recommendations.append({
                    'workload': workload['name'],
                    'eligible': True,
                    'reason': 'Resilient to interruptions',
                    'estimated_savings_pct': estimated_savings_pct,
                    'risk_level': 'low' if workload['max_interruption_rate'] > 0.2 else 'medium'
                })
            else:
                recommendations.append({
                    'workload': workload['name'],
                    'eligible': False,
                    'reason': 'Not resilient to interruptions',
                    'estimated_savings_pct': 0
                })
        
        return recommendations
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 3. Chargeback Models

### Showback & Chargeback Implementation
```python
class ShowbackModel:
    """Transparent cost allocation without actual chargebacks"""
    
    def __init__(self):
        self.allocations = {}
    
    def allocate_costs(self, resources: dict, base_costs: dict) -> dict:
        """Allocate costs to teams/projects"""
        allocations = {}
        total_cost = sum(base_costs.values())
        
        for resource_id, allocation_keys in resources.items():
            team = allocation_keys.get('team', 'unassigned')
            project = allocation_keys.get('project', 'unassigned')
            
            cost = base_costs.get(resource_id, 0)
            
            key = f"{team}/{project}"
            if key not in allocations:
                allocations[key] = 0
            
            allocations[key] += cost
        
        return allocations
    
    def generate_showback_report(self) -> str:
        """Generate cost transparency report"""
        report = "=== SHOWBACK REPORT (For Information Only) ===\n\n"
        report += "This report shows your estimated cloud costs allocation.\n"
        report += "These are informational only and are not actual charges.\n\n"
        return report

class ChargebackModel:
    """Actual cost recovery model"""
    
    def __init__(self, cost_recovery_rate: float = 1.0):
        self.cost_recovery_rate = cost_recovery_rate
        self.invoices = []
    
    def generate_invoice(self, department: str, allocated_cost: float) -> dict:
        """Generate chargeback invoice"""
        chargeable_cost = allocated_cost * self.cost_recovery_rate
        
        invoice = {
            'department': department,
            'allocated_cost': allocated_cost,
            'cost_recovery_rate': self.cost_recovery_rate,
            'chargeable_cost': chargeable_cost,
            'invoice_date': datetime.now(),
            'due_date': datetime.now() + timedelta(days=30)
        }
        
        self.invoices.append(invoice)
        return invoice
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Cost Optimization Checklist

✅ **Cost Tracking**
- [ ] Cost allocation tags implemented
- [ ] Cost center tracking enabled
- [ ] Cloud cost management tool configured (AWS Cost Explorer, GCP Cost Management, Azure Cost Management)
- [ ] Budget alerts configured
- [ ] Cost trends monitored

✅ **Optimization**
- [ ] Idle resources identified and removed
- [ ] Right-sizing analysis performed
- [ ] Reserved instances purchased (>30% savings typical)
- [ ] Committed use discounts evaluated
- [ ] Spot/preemptible instances leveraged (resilient workloads)

✅ **Governance**
- [ ] Cost chargeback model defined
- [ ] Cost owners assigned
- [ ] Budget approval process established
- [ ] Spend limits enforced
- [ ] Regular cost reviews scheduled

✅ **Monitoring**
- [ ] Cost anomalies detected
- [ ] Forecast vs actual variance tracked
- [ ] Cost per unit tracked (cost per request, per GB, etc.)
- [ ] Optimization savings measured
- [ ] FinOps KPIs reported
</MySCode.Cell>
```